# 02: Parameter Optimization

This notebook details the rigorous multi-stage optimization process used to derive the strategy's final parameter set. We prioritize **robustness** and **stability** over raw performance to minimize the risk of curve-fitting whilst increasing the range of regimes in which our strategy survives.

## 1. Parameter Space Overview

The search space consists of 6 core parameters. To ensure validity, we apply a constraint where **Z-entry > Z-exit**, preventing 'instant-flip' trade logic.

### Parameter Ranges:

| Parameter | Range | Description |
| :--- | :--- | :--- |
| **Z-entry** | 1.4 to 3.0 | Threshold for opening a position |
| **Z-exit** | 0.2 to 1.4 | Threshold for mean-reversion exit |
| **Z-stop** | 4.0 to 6.0 | Hard statistical stop-loss |
| **HR Threshold** | 0.2 to 0.8 | Sensitivity of the Hedge Ratio Guard |
| **Resid Lookback** | 30 to 90 days | Lookback for OLS regression parameters |

**Total Combinations:** 1,824 valid sets (after validity constraints).

> **Technical Note:** All Sharpe Ratios referenced in this notebook and used for scoring are **Daily Sharpe Ratios** (non-annualized). This provides a more granular view of risk-adjusted returns across the 15-fold walk-forward period.

## 2. Marginal Sensitivity Analysis

To identify which parameters actually drive performance (load-bearing) and which are noise (non-binding), we perform a marginal analysis.

### Scoring Methodology
For every valid parameter combination, we first calculate an **Aggregate Score** across all 15 walk-forward folds:

$$Score_{combo} = \text{mean}(\text{Daily Sharpe}) - 0.5 \times \text{std}(\text{Daily Sharpe})$$

This function rewards both raw performance and consistency, penalizing parameter sets with high variance across regimes.

### Marginal Analysis Definition
To isolate the impact of a single parameter value (e.g., **Z-entry = 2.2**), we hold that value constant and calculate the **mean of the Aggregate Scores** from all combinations in which that value appears. This "Marginal Score" identifies values that are structurally robust regardless of the other parameter settings.

### Entry Z-Score Sensitivity

![Entry Z](../results/optimization/locked_entry_z_sensitivity.png)

*Observation: Sharpe Ratio peaks clearly at 2.2. Lower values increase trade frequency but degrade quality; higher values lead to trade starvation.*

### Exit Z-Score Sensitivity
![Exit Z](../results/optimization/locked_exit_z_sensitivity.png)

*Observation: A 1.0 exit provides the best balance between Sharpe and total trades.*

### Hedge Ratio Threshold Sensitivity
![HR Thresh](../results/optimization/locked_hr_thresh_sensitivity.png)

*Observation: The guard is remarkably robust. Performance is flat from 0.4 to 1.0, suggesting that any reasonable threshold captures the structural decoupling events. Thresholds under 0.4, however starve the strategy of trades, as seen in 0.2, a guard that tight prevents the any pairs that has minimally shifted from entering trades*

### Residual Lookback Sensitivity
![Resid Val](../results/optimization/locked_resid_val_sensitivity.png)

*Observation: 90 days outperformed shorter windows, suggesting that Mega-cap sector relationships require a longer memory to establish structural stability.*

## 3. Design Decision: Locked vs Unlocked Reference

A key architectural choice was whether to use **Rolling** statistics during a trade or **Locked** statistics (fixed at entry).

| Design | Logic | Performance (IS Sharpe) |
| :--- | :--- | :---: |
| **Unlocked** | Z-score updates with rolling mean/std | 0.42 |
| **Locked** | Z-score computed against entry mean/std | **0.48** |

**Decision: Locked.** Locking prevents the rolling window from 'absorbing' the spread drift, ensuring that the trade only closes when it actually reverts to the original entry thesis.

## 4. The 72-Combination Shortlist

Using the sensitivity topography, we derived a shortlist of 72 combinations (spanning the peaks of Entry Z, Exit Z, and HR Threshold) for full walk-forward evaluation across 15 folds.

In [6]:
import pandas as pd
import numpy as np

# Load actual results from the 72-combination grid search
shortlist = pd.read_csv('../results/optimization/shortlist_72.csv')

print(f"72 combinations analyzed across 15 folds.")
print("Top 10 combinations ranked by Stability Score:")
shortlist[['entry_z', 'exit_z', 'hr_thresh', 'resid_val', 'avg_sharpe', 'std_sharpe', 'score']].head(10)

72 combinations analyzed across 15 folds.
Top 10 combinations ranked by Stability Score:


,entry_z,exit_z,hr_thresh,resid_val,avg_sharpe,std_sharpe,score
0,2.2,1.0,0.8,90,0.106787,0.111558,0.051008
1,2.2,1.0,0.6,90,0.097793,0.104190,0.045698
2,2.2,1.0,0.4,90,0.082407,0.076600,0.044107
3,2.2,0.6,0.8,90,0.105257,0.123188,0.043663
4,1.8,1.4,0.6,45,0.087310,0.092236,0.041192
5,2.2,1.4,0.8,90,0.096225,0.113942,0.039255
6,2.2,0.6,0.6,90,0.101114,0.124467,0.038880
7,2.2,1.4,0.6,90,0.089070,0.102845,0.037648
8,2.2,1.4,0.4,90,0.076337,0.084409,0.034132
9,2.2,0.6,0.4,90,0.080813,0.094103,0.033761


## 5. The Difficult Fold Deep Dive

The centerpiece of our validation was **Fold 11 (Dec 2022 - Mar 2023)**. This period encompassed a regime transition and the SVB banking shock, making it a critical discriminator for robustness.

Most parameter sets that performed well in trending markets failed here. The following table compares how each of the 15 'fold winners' performed when subjected to the conditions of Fold 11.

In [4]:
# Load fold 11 diagnostics comparison
difficult_fold_comp = pd.read_csv('../results/optimization/difficult_fold_results.csv')

print("Performance of all Fold Winners on the 'Difficult Fold' (Fold 11):")
display(difficult_fold_comp.sort_values('Sharpe (Ann)', ascending=False))

Performance of all Fold Winners on the 'Difficult Fold' (Fold 11):


,Winner,F11 Trades,Win Rate,Avg Hold,Sharpe (Ann),Signal,Stop Loss,Guard Trip,Timeout,Max Loss Exit
11,Fold 11,13,0.5385,10.77,0.9894,7,2,0,2,2
4,Fold 4,13,0.5385,9.38,0.3596,7,2,0,2,2
2,Fold 2,11,0.4545,10.73,-0.3049,5,2,0,2,2
3,Fold 3,11,0.4545,10.73,-0.3049,5,2,0,2,2
6,Fold 6,6,0.1667,6.67,-0.7728,1,1,3,0,1
8,Fold 8,16,0.4375,8.38,-1.5585,7,4,2,1,2
12,Fold 12,14,0.4286,8.57,-1.8553,5,4,3,0,2
9,Fold 9,8,0.5000,7.38,-1.9195,3,3,2,0,0
10,Fold 10,4,0.2500,10.25,-2.8138,0,1,2,0,1
5,Fold 5,9,0.3333,9.00,-3.6311,2,5,1,1,0


### Why most sets failed:
Analysis of the trade logs revealed a stark contrast in exit reasons:
- **Failing Sets:** Shorter hedge-ratio estimation windows experienced a higher frequency of stop-loss events during the SVB banking crisis period.
- **The Survivor (Fold 11 Winner):** Dominated by signal-based exits when using a more conservative lookback (90)

## 6. Away Mean Sharpe Comparison

To ensure transferability, we look at **Away Mean Sharpe**—the average performance of a parameter set on all folds *except* the one it was optimized on. This prevents us from selecting a set that was simply 'lucky' in one specific window.

In [5]:
# Load Home/Away Statistics
away_stats = pd.read_csv('../results/optimization/home_away_stats.csv')

print("Fold Winners Ranked by Away Mean Sharpe:")
display(away_stats[['winner_fold', 'home_sharpe', 'away_sharpe_mean', 'away_sharpe_std', 'away_positive_folds']].sort_values('away_sharpe_mean', ascending=False))

Fold Winners Ranked by Away Mean Sharpe:


,winner_fold,home_sharpe,away_sharpe_mean,away_sharpe_std,away_positive_folds
3,fold_11,0.0623,0.1100,0.1151,12
7,fold_2,0.1861,0.0950,0.1268,9
8,fold_3,0.2690,0.0891,0.1198,9
10,fold_5,0.2091,0.0856,0.1865,10
9,fold_4,0.2522,0.0851,0.1094,10
4,fold_12,0.1000,0.0777,0.1332,9
5,fold_13,0.3940,0.0723,0.1687,10
2,fold_10,0.2562,0.0708,0.1365,10
13,fold_8,0.4081,0.0594,0.1325,9
6,fold_14,0.1493,0.0573,0.1820,10


### Selection Criterion:
We prioritize the **highest away mean Sharpe** over the highest individual home fold Sharpe. The Fold 11 winner achieved the top spot here, confirming that its robustness in the difficult fold translates to consistent performance across the entire 2020-2024 period.

## 7. Parameter Set Convergence

The final parameter selection was not the result of a single metric, but rather the convergence of four criteria:

1. **Away Mean Sharpe:** The Fold 11 set independently ranked #1 in average performance across all folds.
2. **Difficult Fold Survival:** It was one of only two sets to maintain a positive Sharpe during the banking shock regime.
3. **Grid Search Stability:** The parameters (Z-entry=2.2, Z-exit=1.0) sat at the absolute peak of the marginal sensitivity topography.
4. **Highest Score:** On top of all the above, the final parameter combination also had the best score amongst all 72 iterations.

This convergence gives us high confidence that the strategy's edge is structural and likely to persist in Out-of-Sample environments.

**Final parameter set chosen:** z-entry = 2.2, z-exit = 1.0, hr_thresh = 0.8, resid_val = 90, z_multiplier = 0.5

The next notebook, **[03: Walk-Forward Results](03_walk_forward_results.ipynb)**, provides a detailed reporting of the performance of this 'Golden Regime' across all validation folds.